In [279]:
import pandas as pd
import numpy as np
import re
import difflib
from cleaning import extract_refresh_rate
from cleaning import extract_camera_info
from cleaning import extract_architecture
from cleaning import add_ram
from cleaning import clean_phone_name
from cleaning import clean_metrics
from cleaning import clean_price
from cleaning import clean_storage
from cleaning import clean_chipset
from cleaning import add_chipset_info
from cleaning import fill_mean_nearest
from cleaning import fill_mean

In [280]:
cps = pd.read_csv(r'cellphones_full.csv')
cps.info()

<class 'pandas.DataFrame'>
RangeIndex: 966 entries, 0 to 965
Data columns (total 19 columns):
 #   Column                 Non-Null Count  Dtype
---  ------                 --------------  -----
 0   Tên                    966 non-null    str  
 1   Giá                    966 non-null    str  
 2   Link                   966 non-null    str  
 3   Kích thước màn hình    865 non-null    str  
 4   Công nghệ màn hình     806 non-null    str  
 5   Camera sau             847 non-null    str  
 6   Camera trước           817 non-null    str  
 7   Chipset                850 non-null    str  
 8   Công nghệ NFC          763 non-null    str  
 9   Bộ nhớ trong           913 non-null    str  
 10  Thẻ SIM                695 non-null    str  
 11  Hệ điều hành           756 non-null    str  
 12  Độ phân giải màn hình  657 non-null    str  
 13  Tính năng màn hình     725 non-null    str  
 14  Loại CPU               586 non-null    str  
 15  Dung lượng RAM         870 non-null    str  
 16  P

ĐỐI TÊN THUỘC TÍNH

In [281]:
feature_mapping = {
    "Tên": "Name",
    "Giá": "Price",
    "Link": "Link",
    "Kích thước màn hình": "Screen Size",
    "Công nghệ màn hình": "Display",
    "Camera sau": "Rear Camera",
    "Camera trước": "Front Camera",
    "Chipset": "Chipset",
    "Công nghệ NFC": "NFC",
    "Bộ nhớ trong": "ROM",
    "Thẻ SIM": "SIM Card",
    "Hệ điều hành": "Operating System",
    "Độ phân giải màn hình": "Screen Resolution",
    "Tính năng màn hình": "Display Features",
    "Loại CPU": "CPU",
    "Dung lượng RAM": "RAM",
    "Pin": "Battery",
    "Tương thích": "Compatibility",
    "Cảm biến": "Sensors",
}

cps = cps.rename(columns=feature_mapping)


In [324]:
df = cps.copy()
df[df['Name'] == 'honor x9d']['Display Features']

19    100% DCI-P3, 1,07 tỷ màu800 nits (điển hình), ...
79    100% DCI-P3, 1,07 tỷ màu800 nits (điển hình), ...
Name: Display Features, dtype: str

Đọc từ gsm và antutu

In [283]:
gsm = pd.read_csv(r'all_phones_final.csv')
gsm = gsm[['name_clean', 'Memory | Internal']]
gsm.info()

<class 'pandas.DataFrame'>
RangeIndex: 799 entries, 0 to 798
Data columns (total 2 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   name_clean         799 non-null    str  
 1   Memory | Internal  590 non-null    str  
dtypes: str(2)
memory usage: 12.6 KB


In [284]:
antutu = pd.read_csv(r'antutu_socket.csv')
att = antutu.copy()
att.info()

<class 'pandas.DataFrame'>
RangeIndex: 233 entries, 0 to 232
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   Link          233 non-null    str  
 1   Chipset       233 non-null    str  
 2   Antutu_11     233 non-null    int64
 3   Clock         233 non-null    str  
 4   GPU           233 non-null    str  
 5   Architecture  233 non-null    str  
dtypes: int64(1), str(5)
memory usage: 11.1 KB


EXTRACT REFRESH RATE

In [285]:
df["Refresh Rate"] = df["Display Features"].apply(extract_refresh_rate)
df["Refresh Rate"].info()

<class 'pandas.Series'>
RangeIndex: 966 entries, 0 to 965
Series name: Refresh Rate
Non-Null Count  Dtype  
--------------  -----  
569 non-null    float64
dtypes: float64(1)
memory usage: 7.7 KB


CLEAN NAME

In [286]:
df["Name"] = df["Name"].apply(clean_phone_name)
gsm['name_clean'] = gsm['name_clean'].apply(clean_phone_name)

In [287]:
def get_brand(text):    
    brand = text.strip().split()[0]
    
    if 'iphone' in brand:
        return 'apple'
    if 'red' in brand:
        return 'nubia'
    if 'xperia' in brand:
        return 'sony'
    if 'black' in brand or 'xiaomii' in brand:
        return 'xiaomi'
    if 'rog' in brand or 'zenfone' in brand:
        return 'asus'
    if 'camon' in brand:
        return 'tecno'
    
    return brand

df['Brand'] = df['Name'].apply(get_brand)
df[['Name','Brand']].head()

,Name,Brand
0,iphone 17,apple
1,oppo find x9s,oppo
2,iphone 17 promax,apple
3,samsung galaxy s26,samsung
4,samsung galaxy s26,samsung


In [288]:
df['Brand'].unique()

<StringArray>
[   'apple',     'oppo',  'samsung',     'itel',    'nubia',   'xiaomi',
    'honor',    'meizu',     'poco',    'tecno',      'zte',  'nothing',
     'sony',   'huawei',    'nokia',  'masstel',  'viettel',   'realme',
     'vivo',     'asus',   'google',  'infinix',   'lenovo',  'oneplus',
   'vsmart',   'bphone',    'benco',    'sharp',    'leitz',     'iqoo',
 'motorola',      'htc',   'doogee',      'tcl']
Length: 34, dtype: str

ADD RAM

In [289]:
df = add_ram(gsm, df)

In [290]:
df['RAM'].info()

<class 'pandas.Series'>
RangeIndex: 966 entries, 0 to 965
Series name: RAM
Non-Null Count  Dtype
--------------  -----
891 non-null    str  
dtypes: str(1)
memory usage: 7.7 KB


CLEAN CHIPSET AND ADD CHIPSET INFO

In [291]:
df['Chipset'] = df['Chipset'].apply(clean_chipset)
df = add_chipset_info(att, df)

In [292]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 966 entries, 0 to 965
Data columns (total 25 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Name               966 non-null    str    
 1   Price              966 non-null    str    
 2   Link               966 non-null    str    
 3   Screen Size        865 non-null    str    
 4   Display            806 non-null    str    
 5   Rear Camera        847 non-null    str    
 6   Front Camera       817 non-null    str    
 7   Chipset            834 non-null    str    
 8   NFC                763 non-null    str    
 9   ROM                913 non-null    str    
 10  SIM Card           695 non-null    str    
 11  Operating System   756 non-null    str    
 12  Screen Resolution  657 non-null    str    
 13  Display Features   725 non-null    str    
 14  CPU                586 non-null    str    
 15  RAM                891 non-null    str    
 16  Battery            861 non-null    st

CLEAN PRICE

In [293]:
df["Price"] = df["Price"].apply(clean_price)

In [294]:
df["Price"].info()

<class 'pandas.Series'>
RangeIndex: 966 entries, 0 to 965
Series name: Price
Non-Null Count  Dtype  
--------------  -----  
337 non-null    float64
dtypes: float64(1)
memory usage: 7.7 KB


CLEAN STORAGE

In [295]:
df["RAM"] = df["RAM"].apply(clean_storage)
df["ROM"] = df["ROM"].apply(clean_storage)

In [296]:
df[["RAM", "ROM"]].info()

<class 'pandas.DataFrame'>
RangeIndex: 966 entries, 0 to 965
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   RAM     891 non-null    float64
 1   ROM     913 non-null    float64
dtypes: float64(2)
memory usage: 15.2 KB


CLEAN METRICS

In [297]:
cols_to_clean = ["Screen Size","Battery", "clock"]
for col in cols_to_clean:
    df[col] = df[col].apply(clean_metrics)

df['Battery'] = df['Battery'].where(df['Battery'] > 100)

In [298]:
df[["Screen Size","Battery", "clock"]].info()

<class 'pandas.DataFrame'>
RangeIndex: 966 entries, 0 to 965
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Screen Size  865 non-null    float64
 1   Battery      679 non-null    float64
 2   clock        747 non-null    float64
dtypes: float64(3)
memory usage: 22.8 KB


EXTRACT ARCHITECTURE

In [299]:
new_columns = ['total_cores', 'min_freq', 'mean_freq']
df[new_columns] = df['architecture'].apply(lambda x : pd.Series(extract_architecture(x)))

df[new_columns].info()

<class 'pandas.DataFrame'>
RangeIndex: 966 entries, 0 to 965
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   total_cores  747 non-null    float64
 1   min_freq     747 non-null    float64
 2   mean_freq    747 non-null    float64
dtypes: float64(3)
memory usage: 22.8 KB


CLEAN OPERATING SYSTEM

In [300]:
def advanced_clean_os(text):
    if (
        pd.isna(text)
        or not isinstance(text, str)
        or "cập nhật" in text.lower()
    ):
        return 1, "Android", None

    text = text.strip()
    
    is_android = 1
    os_name = "Android"
    if "ios" in text.lower():
        is_android = 0
        os_name = "iOS"

    # trích xuất số đời (Version)
    os_version = None

    # Trường hợp A: Dòng chỉ chứa mỗi số (Ví dụ: '11')
    if text.isdigit():
        return is_android, os_name, float(text)

    # Trường hợp B: Dòng phức tạp có chữ 'có thể nâng cấp lên Android X'
    if "nâng cấp" in text.lower():
        upgraded_version = re.findall(r"Android\s*(\d+(?:\.\d+)?)", text)
        if upgraded_version:
            # Lấy số phiên bản cuối cùng (cao nhất) trong chuỗi
            return is_android, os_name, float(upgraded_version[-1])

    # Trường hợp C: Dòng thông thường, tìm số đi ngay sau chữ 'Android' hoặc 'iOS'
    version_match = re.search(r"(?:Android|iOS)\s*(\d+(?:\.\d+)?)", text, re.I)
    if version_match:
        os_version = float(version_match.group(1))
    else:
        # Trường hợp như không có số, tạm để None hoặc gán số 8.0/9.0
        os_version = None

    return is_android, os_name, os_version

In [301]:
df["OS_Is_Android"], df["OS_Name"], \
    df["OS_Version"] = zip(*df["Operating System"].apply(advanced_clean_os))

CLEAN RESOLUTION

In [302]:
def extract_res_row(text):
    # Nếu dòng bị trống (NaN) hoặc không phải chữ
    if pd.isna(text) or not isinstance(text, str):
        return None, None

    # Tìm cấu trúc a x b ở đầu dòng
    match = re.search(r"^(\d+)\s*[xX×]\s*(\d+)", text.strip())

    if match:
        # Trả về một Tuple gồm (Width, Height) kiểu số nguyên
        return int(match.group(1)), int(match.group(2))

    return None, None

In [303]:
df["Reso_Width"], df["Reso_Height"] = zip(
    *df["Screen Resolution"].apply(extract_res_row)
)

In [304]:
def clean_sim_options(text):
    max_nano = 0
    max_esim = 0
    max_micro = 0
    max_mini = 0

    if pd.isna(text) or not isinstance(text, str):
        return max_nano, max_esim, max_micro, max_mini

    text_lower = text.lower()

    options = re.split(r"hoặc|/|;", text_lower)

    for option in options:
        option = option.strip()

        nano_in_opt = 0
        esim_in_opt = 0

        # Xử lý eSIM 
        if "esim" in option:
            match_esim = re.search(r"(\d+)\s*esim", option)
            if match_esim:
                esim_in_opt = int(match_esim.group(1))
            elif "dual" in option or "kép" in option:
                esim_in_opt = 2
            else:
                esim_in_opt = 1

        # Xử lý Nano SIM 
        if "nano" in option or "sim 1 + sim 2" in option:
            match_nano = re.search(r"(\d+)\s*nano", option)
            if match_nano:
                nano_in_opt = int(match_nano.group(1))
            elif (
                "dual" in option
                or "kép" in option
                or "sim 1 + sim 2" in option
            ):
                nano_in_opt = 2
            else:
                nano_in_opt = 1
        elif "2 sim" in option and "nano" in text_lower:
            nano_in_opt = 2
        # Trường hợp ghi mỗi chữ "Nano-SIM" thuần túy
        elif "nano" in option:
            nano_in_opt = 1

        # Cập nhật giá trị max
        max_nano = max(max_nano, nano_in_opt)
        max_esim = max(max_esim, esim_in_opt)

        # Xử lý các loại SIM cổ (Mini, Micro)
        if "micro" in option:
            max_micro = 1
        if "mini" in option:
            max_mini = 1

    return max_nano, max_esim, max_micro, max_mini

In [305]:
(
    df["Nano_SIM_Count"],
    df["eSIM_Count"],
    df["Micro_SIM_Count"],
    df["Mini_SIM_Count"],
) = zip(*df["SIM Card"].apply(clean_sim_options))

CLEAN NFC

In [306]:
df['NFC'] = df['NFC'].map(lambda x : 1 if x == "Có" else 0)

CLEAN CAMERA

In [307]:
df = extract_camera_info(df)

In [308]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 966 entries, 0 to 965
Data columns (total 45 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Name               966 non-null    str    
 1   Price              337 non-null    float64
 2   Link               966 non-null    str    
 3   Screen Size        865 non-null    float64
 4   Display            806 non-null    str    
 5   Rear Camera        847 non-null    str    
 6   Front Camera       817 non-null    str    
 7   Chipset            834 non-null    str    
 8   NFC                966 non-null    int64  
 9   ROM                913 non-null    float64
 10  SIM Card           695 non-null    str    
 11  Operating System   756 non-null    str    
 12  Screen Resolution  657 non-null    str    
 13  Display Features   725 non-null    str    
 14  CPU                586 non-null    str    
 15  RAM                891 non-null    float64
 16  Battery            679 non-null    fl

In [309]:
bool_cols = ['rear_ois', 'rear_telephoto', 'rear_wide']
for col in bool_cols:
    df[col] = df[col].fillna(0).astype(bool)

In [310]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 966 entries, 0 to 965
Data columns (total 45 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Name               966 non-null    str    
 1   Price              337 non-null    float64
 2   Link               966 non-null    str    
 3   Screen Size        865 non-null    float64
 4   Display            806 non-null    str    
 5   Rear Camera        847 non-null    str    
 6   Front Camera       817 non-null    str    
 7   Chipset            834 non-null    str    
 8   NFC                966 non-null    int64  
 9   ROM                913 non-null    float64
 10  SIM Card           695 non-null    str    
 11  Operating System   756 non-null    str    
 12  Screen Resolution  657 non-null    str    
 13  Display Features   725 non-null    str    
 14  CPU                586 non-null    str    
 15  RAM                891 non-null    float64
 16  Battery            679 non-null    fl

Encode OS name. 1: Android and 0: iOS

In [311]:
df['OS'] = (df['OS_Name'] == 'iOS').astype(int) #iOS = 1 | else = 0

Encode Display. 1: OLED/AMOLED, 2: IPS LCD/IPS/LCD, 0: other

In [312]:
display_lower = df['Display'].astype(str).str.lower()

conditions = [
    display_lower.str.contains('oled|amoled', regex=True),  # Nhóm 2
    display_lower.str.contains('ips lcd|ips|lcd', regex=True)      # Nhóm 1
]

choices = [2, 1]

df['Display'] = np.select(conditions, choices, default=0)

Encode Chipset. Apple: 0, Snapdragon: 1, MediaTek: 2, Exynos: 3, Kirin: 4, Unisoc: 5, Other: 6

In [313]:
def encode_chipset(val):
    val = str(val).lower()
    if 'apple' in val or 'chip a' in val or 'bionic' in val:
        return 'Apple'
    elif 'snapdragon' in val or 'qualcomm' in val:
        return 'Snapdragon'
    elif 'dimensity' in val or 'helio' in val or 'mediatek' in val:
        return 'MediaTek'
    elif 'exynos' in val:
        return 'Exynos'
    elif 'kirin' in val:
        return 'Kirin'
    elif 'unisoc' in val:
        return 'Unisoc'
    else:
        return 'Other'
 
brand_map = {'Apple': 0, 'Snapdragon': 1, 'MediaTek': 2,
             'Exynos': 3, 'Kirin': 4, 'Unisoc': 5, 'Other': 6}
df['Chipset'] = df['Chipset'].apply(encode_chipset).map(brand_map)

Encode GPU

In [314]:
def encode_gpu(val):
    val = str(val).lower()
    if 'adreno'     in val: return 'Adreno'
    elif 'mali'     in val: return 'Mali'
    elif 'apple'    in val: return 'Apple GPU'
    elif 'xclipse'  in val: return 'Xclipse'
    elif 'powervr'  in val or 'img' in val: return 'PowerVR'
    elif 'immortalis' in val: return 'Immortalis'
    elif 'maleoon'  in val: return 'Maleoon'
    else: return 'Other'
 
gpu_map = {'Adreno': 0, 'Mali': 1, 'Apple GPU': 2, 'Xclipse': 3,
           'PowerVR': 4, 'Immortalis': 5, 'Maleoon': 6, 'Other': 7}
df['gpu_family_enc'] = df['gpu'].apply(encode_gpu).map(gpu_map)

Derive PPI

In [315]:
df['PPI'] = (np.sqrt(df['Reso_Width']**2 + df['Reso_Height']**2) / df['Screen Size']).round(1)
df.drop(columns=['Reso_Width', 'Reso_Height'], inplace=True)

Derive SIM_total

In [316]:
df['SIM_total'] = (df['Nano_SIM_Count'] + df['eSIM_Count'] + df['Micro_SIM_Count'] + df['Mini_SIM_Count'])

df.drop(columns=['Nano_SIM_Count', 'Micro_SIM_Count', 'Mini_SIM_Count'], inplace=True)

Derive eSIM, dien thoai nao co eSIM: 1, khong co: 0

In [317]:
df['has_eSIM'] = (df['eSIM_Count'] > 0).astype(int)

df.drop(columns=['eSIM_Count'], inplace=True)

In [318]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 966 entries, 0 to 965
Data columns (total 44 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Name               966 non-null    str    
 1   Price              337 non-null    float64
 2   Link               966 non-null    str    
 3   Screen Size        865 non-null    float64
 4   Display            966 non-null    int64  
 5   Rear Camera        847 non-null    str    
 6   Front Camera       817 non-null    str    
 7   Chipset            966 non-null    int64  
 8   NFC                966 non-null    int64  
 9   ROM                913 non-null    float64
 10  SIM Card           695 non-null    str    
 11  Operating System   756 non-null    str    
 12  Screen Resolution  657 non-null    str    
 13  Display Features   725 non-null    str    
 14  CPU                586 non-null    str    
 15  RAM                891 non-null    float64
 16  Battery            679 non-null    fl

In [319]:
drop = ['Link', 'Rear Camera', 'Front Camera', 'CPU', 'Display Features', 'Screen Resolution',
    'Operating System', 'SIM Card', 'OS_Is_Android', 'Compatibility', 'Sensors', 'OS_Version', 'architecture']

In [320]:
df.drop(columns=drop, inplace = True)

In [321]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 966 entries, 0 to 965
Data columns (total 31 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Name            966 non-null    str    
 1   Price           337 non-null    float64
 2   Screen Size     865 non-null    float64
 3   Display         966 non-null    int64  
 4   Chipset         966 non-null    int64  
 5   NFC             966 non-null    int64  
 6   ROM             913 non-null    float64
 7   RAM             891 non-null    float64
 8   Battery         679 non-null    float64
 9   Refresh Rate    569 non-null    float64
 10  Brand           966 non-null    str    
 11  antutu_11       747 non-null    float64
 12  clock           747 non-null    float64
 13  gpu             747 non-null    str    
 14  total_cores     747 non-null    float64
 15  min_freq        747 non-null    float64
 16  mean_freq       747 non-null    float64
 17  OS_Name         966 non-null    str    
 18  r

In [322]:
df.to_csv('full_data_2.csv')